# MDELAN: channel-estimation MSE, LEO NTN rural

Reproduces the rural MSE vs. SNR figure (`MSE_MCS9_rural.png`) of the MDELAN paper
(arXiv:2607.15127, SPAWC 2026): QuaDRiGa LEO NTN rural dataset, 2x1 SIMO, 1 user, 11 PRBs, MCS 9.

The `*_results` files in this folder are produced by `run_rural.sh`.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # suppress TF warnings

import sys
sys.path.append("../../")  # repository root

import pickle

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from ext.neural_rx.utils.parameters import Parameters
from ext.neural_rx.utils.utils import plot_results
from utils.eval_plots import COLORMAP, plot_error_metric_results

results_dir = "./"  # this folder
num_tx_eval, mcs_arr_eval_idx = 1, 0  # 1 user, MCS 9

# legend label -> style [linestyle, COLORMAP index, marker, marker size]
styles = {
    "LS":                    ["-",  2, "o", 4],
    "LMMSE":                 ["--", 4, "^", 4],
    "MDX":                   ["-",  8, "s", 4],
    "MDX: fine tuned":       [":",  8, "s", 4],
    "MDX:MDELAN":            ["-",  0, "v", 4],
    "MDX:MDELAN fine-tuned": [":",  0, "v", 4],
}


def plot_mse(ax, config_name, suffix, label, system, xlim, ylim):
    """Plot one MSE curve read from ``<config label><suffix>_results``.

    ``system`` picks the series by receiver name when a file holds several.
    The released files use the legacy list layout, with the MSE in the BER slot;
    files re-generated by ``run_rural.sh`` use the current dict layout.
    """
    config_label = Parameters(config_name, training=False, system="dummy").label
    with open(f"{results_dir}{config_label}{suffix}_results", "rb") as f:
        data = pickle.load(f)
    legacy = isinstance(data, (list, tuple))
    keys = [k for k in (data[1] if legacy else data["mse"])
            if k[1] == num_tx_eval and k[2] == mcs_arr_eval_idx]
    selected = [system is None or k[0] == system for k in keys]
    kwargs = dict(
        config_name=config_name,
        num_tx_eval=num_tx_eval,
        mcs_arr_eval_idx=mcs_arr_eval_idx,
        xlim=xlim,
        ylim=ylim,
        labels=[label if s else f"-{k[0]}" for k, s in zip(keys, selected)],  # "-" skips a series
        line_styles=[styles[label] if s else "-" for s in selected],
        results_dir=[results_dir, suffix],
        axis=ax,
        show=[False, False, False, False, True, True],  # only the axis ticks
    )
    if legacy:
        plot_results(show_ber=True, **kwargs)
    else:
        plot_error_metric_results(metric="mse", **kwargs)


def plot_figure(scenario, curves, xlim, ylim=(8e-4, 2e0)):
    """Draw ``curves`` ({label: (config, suffix, system)}) and save MSE_MCS9_<scenario>.png."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    for label, (config_name, suffix, system) in curves.items():
        plot_mse(ax, config_name, suffix, label, system, xlim, ylim)
    ax.set_xlabel(r"$\mathrm{SNR (dB)}$")
    ax.set_ylabel("MSE")
    legend_elements = [
        Line2D([0], [0], linewidth=2, color=COLORMAP[s[1]], linestyle=s[0], marker=s[2], label=label)
        for label, s in styles.items()
    ]
    ax.legend(handles=legend_elements, loc="lower center", bbox_to_anchor=(0.5, 1), ncol=3, fontsize=9)
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    fig.savefig(f"MSE_MCS9_{scenario}.png", dpi=600, bbox_inches="tight")
    plt.show()

In [ ]:
# NTN rural
plot_figure("rural", {
    "LS":                    ("baselines_2x1.cfg",           "_rural",        "Baseline - LS/lin+LMMSE"),
    "LMMSE":                 ("baselines_2x1.cfg",           "_rural",        "Baseline - LMMSE+K-Best"),
    "MDX":                   ("mdx_ntn.cfg",                 "_original_new", None),
    "MDX: fine tuned":       ("mdx_ntn_frural.cfg",          "_original_new", None),
    "MDX:MDELAN":            ("mdx_ntn_mdelan2f.cfg",        "_new",          None),
    "MDX:MDELAN fine-tuned": ("mdx_ntn_mdelan2f_frural.cfg", "_new",          None),
}, xlim=[-10, 10])